In [ ]:
# 01 · CONFIG · Colab 2026.07 / Python 3.12 / T4
# 이전 실험과 다른 run_name입니다. 가중 표집은 새로 학습해야 적용됩니다.
# 모델은 [32, 64, 128], batch 32 유지. 평가 제한 200스텝 유지.
CFG = {
    # GitHub 결과 · 같은 run_name은 저장 모델 재사용, 새 학습은 새 run_name
    "run_name": 'moveboxes_next_pick_v02',
    "profile": 'benchmark',
    "output_root": '/content/moveboxes_runs',
    "data_source": '/content/moveboxes_data_cache/marso_state_data.zip',

    # 공식 코드 · 런타임 2026.07 / T4
    "repo_dir": '/content/berlin-marso-hackathon',
    "data_dir": '/content/marso_data',
    "repo_url": 'https://github.com/marso-robotics/berlin-marso-hackathon.git',
    "repo_commit": '6048f33217f26ae39009a812f53c81171517f393',
    "packages": [
        'mani-skill==3.0.1',
        'sapien==3.0.3',
        'diffusers==0.38.0',
        'hydra-core',
        'omegaconf',
        'gymnasium',
        'tyro',
        'h5py',
        'kagglehub',
        'tensorboard',
        'matplotlib',
        'transforms3d',
        'imageio[ffmpeg]',
    ],

    # 학습 · T4용 소형 모델, batch 32
    "seed": 42,
    "num_demos": None,
    "batch_size": 32,
    "lr": 0.0001,
    "total_iters": {'easy': 30000, 'medium': 50000, 'hard': 60000},

    # 모델 · 새 학습에 적용; 기존 checkpoint 구조는 저장 설정 사용
    "obs_horizon": 2,
    "pred_horizon": 16,
    "act_horizon": 4,
    "unet_dims": [32, 64, 128],
    "diffusion_step_embed_dim": 32,
    "n_groups": 8,

    # 학습 중 평가 / 저장
    "eval_freq": 10000,
    "save_freq": 10000,
    "log_freq": 500,
    "train_eval_episodes": 4,
    "num_eval_envs": 1,

    # 평가 · 난이도별 제한 200스텝 유지
    "max_episode_steps": {'easy': 200, 'medium': 200, 'hard': 200},
    "tuning_episodes": 8,
    "tuning_seed_start": 20000,
    "chunk_candidates": [4, 8],
    "denoising_candidates": [16, 32],
    "max_checkpoints": 2,
    "benchmark_episodes": 100,
    "eval_seed_start": 30000,

    # 출력
    "record_eval_video": True,
    "console_interval_seconds": 30,
    "team": 'my-team',


    # 빠른 테스트 · 최종 평가와 별도 저장
    "test_episodes": 8,
    "test_seed_start": 40000,
    "test_inference_steps": 16,
    "test_record_video": True,

    # 다음 집기 집중 학습 · 2번째 이후 안정적인 집기 사이클 주변 시연을 3배 가중 표집
    # 같은 상자의 재집기도 포함될 수 있음. sampling_audit.json에서 탐지 결과 확인.
    # 비교 실험: 새 run_name을 쓰고 focus_sampling=False, act_horizon=8로 설정.
    "focus_sampling": True,
    "focus_weight": 3.0,
    "focus_before_steps": 20,
    "focus_after_steps": 8,
    "focus_min_grasp_steps": 3,
    "focus_min_release_steps": 3,

    # GitHub · 토큰은 CONFIG에 쓰지 않고 Colab 보안 비밀 GH_TOKEN에 등록
    "github_repository": 'SongYunu/moveBoxes',
    "project_dir": '/content/moveBoxes',
    "project_ref": 'main',
    "download_cache": '/content/moveboxes_data_cache',
}


In [ ]:
# 02 · GitHub 코드 불러오기 (데이터·결과를 위해 Drive를 마운트하지 않습니다)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
URL = 'https://github.com/'+CFG['github_repository']+'.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if remote != URL:
        raise RuntimeError('기존 프로젝트 폴더가 다른 저장소입니다. project_dir를 새 경로로 바꾸세요.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CFG['project_ref']], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
CFG['project_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
sys.path.insert(0, str(PROJECT))
# A fresh notebook run should not retain a previously imported project module.
for name in ('marso_experiment', 'marso_train_test', 'next_pick_sampling', 'next_pick_diagnostics',
             'marso_next_pick', 'github_store', 'github_data', 'marso_github'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
from marso_github import GitHubExperiment, source_bundle
experiment = GitHubExperiment(CFG, source_bundle())
print('사용 코드:', CFG['project_commit'])


In [ ]:
# 03 · GitHub 인증 / 저장된 결과 복원
# 왼쪽 열쇠 아이콘 → GH_TOKEN 추가 → 노트북 액세스 허용.
# Fine-grained token: SongYunu/moveBoxes → Contents: Read and write.
# 이 저장소는 공개이므로 여기에 올린 모델·로그·영상도 공개됩니다.
from google.colab import userdata
try:
    os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    raise RuntimeError('Colab 보안 비밀에 GH_TOKEN을 등록하고 노트북 액세스를 허용하세요.') from None
experiment.connect()
experiment.show_results()


In [ ]:
# 04 · 새 런타임마다 환경 설치
experiment.install()


In [ ]:
# 05 · GitHub 데이터 다운로드·검증 / GPU와 정책 실행 확인
experiment.prepare_data()
experiment.check_runtime()


In [ ]:
# 06 · EASY 동작 확인: 별도 폴더에서 100 iteration + 2회 시험
# 성능 점수가 아니라 학습·모델 로딩·환경 실행 확인입니다.
experiment.smoke("easy")


In [ ]:
# 07 · EASY 전체 학습 · 다음 집기 구간 가중 표집 → GitHub
experiment.train("easy")


In [ ]:
# 08 · EASY 빠른 테스트: 8회 + 영상 + 다음 집기 진단 → test_metrics.json
experiment.test("easy")


In [ ]:
# 09 · EASY 설정 비교 + 최종 100회 평가 → metrics.json
# 시간이 오래 걸릴 수 있습니다. 필요할 때 이 셀만 따로 실행할 수 있습니다.
experiment.evaluate("easy")


In [ ]:
# 10 · MEDIUM 동작 확인: 별도 폴더에서 100 iteration + 2회 시험
# 성능 점수가 아니라 학습·모델 로딩·환경 실행 확인입니다.
experiment.smoke("medium")


In [ ]:
# 11 · MEDIUM 전체 학습 · 다음 집기 구간 가중 표집 → GitHub
experiment.train("medium")


In [ ]:
# 12 · MEDIUM 빠른 테스트: 8회 + 영상 + 다음 집기 진단 → test_metrics.json
experiment.test("medium")


In [ ]:
# 13 · MEDIUM 설정 비교 + 최종 100회 평가 → metrics.json
# 시간이 오래 걸릴 수 있습니다. 필요할 때 이 셀만 따로 실행할 수 있습니다.
experiment.evaluate("medium")


In [ ]:
# 14 · HARD 동작 확인: 별도 폴더에서 100 iteration + 2회 시험
# 성능 점수가 아니라 학습·모델 로딩·환경 실행 확인입니다.
experiment.smoke("hard")


In [ ]:
# 15 · HARD 전체 학습 · 다음 집기 구간 가중 표집 → GitHub
experiment.train("hard")


In [ ]:
# 16 · HARD 빠른 테스트: 8회 + 영상 + 다음 집기 진단 → test_metrics.json
experiment.test("hard")


In [ ]:
# 17 · HARD 설정 비교 + 최종 100회 평가 → metrics.json
# 시간이 오래 걸릴 수 있습니다. 필요할 때 이 셀만 따로 실행할 수 있습니다.
experiment.evaluate("hard")


In [ ]:
# 18 · GitHub의 테스트·최종 평가 결과와 영상 보기
experiment.show_tests(videos=True)
experiment.show_results(videos=True)


In [ ]:
# 19 · 최종 평가까지 끝난 모델을 GitHub에 패키징
experiment.package()

print("원격 결과:", "https://github.com/"+CFG["github_repository"]+"/releases")
